# Mock Interview Scenarios: Customer Scenarios (Full Timed Simulations)

**Anthropic Applied AI Engineer -- Customer Scenarios Interview**

---

## What This Notebook Is

This notebook contains **3 complete timed practice runs** simulating the full 50-minute Customer Scenarios interview at Anthropic. In this interview, the interviewers role-play as a technical stakeholder or customer. Your job is to:

1. **Discovery** (~15 min): Collect as much info as possible, listen to concerns, build trust
2. **Solo Build** (~15-20 min): Extend starter code (screen shared) into a working solution -- structured I/O, reliable classifier
3. **Present** (~15-20 min): Walk the customer through what you built, why it works, and how it reduces risk

**They give you starter code.** You're extending, not starting from scratch.

**Sweet spot**: Structured input/output, reliable classifiers with confidence scoring.

## How to Use This Notebook

For each scenario:

1. **Set a timer** -- use the phase timings listed before each scenario
2. **Read the customer brief** -- this is what the "customer" would say to open the interview
3. **Write your discovery questions** in the empty cells BEFORE scrolling to the reference questions
4. **Build your solution** in the empty cells BEFORE scrolling to the reference solution
5. **Practice your presentation out loud** -- explain what you built, why, and how it reduces risk
6. **Compare** against the reference solution and self-evaluate

**Treat this like game day.** Don't peek ahead. The empty cells are your test.

---

| Phase | Time | What You Do |
|-------|------|-------------|
| **1. Discovery** | ~15 min | Ask questions, understand problem + concerns (write concerns down!) |
| **2. Solo Build** | ~15-20 min | Extend starter code: system prompt + structured output + test cases |
| **3. Present** | ~15-20 min | Walk through solution, demo live, address every concern, production roadmap |

---

## Setup

Run this cell first to set up the API client and helper functions.

In [ ]:
!pip install anthropic -q
from google.colab import userdata
import anthropic, json, time

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

def run(prompt, system="You are a helpful assistant.", model="claude-haiku-4-5-20251001", max_tokens=4096):
    """Quick helper to call Claude."""
    response = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=system, messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

def run_multi(messages, system="You are a helpful assistant.", model="claude-haiku-4-5-20251001", max_tokens=4096):
    """Multi-turn helper."""
    response = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=system, messages=messages
    )
    return response.content[0].text

print("Setup complete. Ready to begin mock interviews.")

---
---

# SCENARIO 1: Education Nonprofit -- Course Feedback Analyzer

---

## Customer Brief

> *The interviewer, playing the role of the customer, says:*

**"Hi, I'm the Director of Programs at LearnForward, a nonprofit that runs after-school coding programs for underserved youth in 15 cities. We collect feedback surveys after every 8-week session -- from students, parents, and instructors. Right now our program team manually reads through hundreds of responses to write quarterly reports for our funders. It takes weeks. We heard AI might help us process this faster. Can you show us what's possible?"**

## Timer Guidance

Set your timer now:

| Phase | Duration | What to Do |
|-------|----------|------------|
| **Phase 1: Discovery** | ~15 minutes | Write discovery questions + note customer concerns |
| **Phase 2: Solo Build** | ~15-20 minutes | Extend starter code: system prompt, structured output, test on samples |
| **Phase 3: Present** | ~15-20 minutes | Walk through solution, demo live, address concerns, risk reduction |

**START YOUR TIMER NOW. Do not scroll past each phase until your time is up.**

**IMPORTANT**: During Discovery, write down every concern the customer mentions. You MUST address each one in Phase 3.

---

## Phase 1: Discovery (10 minutes)

Write your discovery questions below. Think about what you need to know before you can build anything. Do NOT look at the reference questions until you have written your own.

In [ ]:
# PHASE 1: DISCOVERY (10 minutes)
# Write your discovery questions here before looking at the reference questions below
#
# Think about:
# - What data format are the surveys in?
# - What goes into the quarterly reports?
# - Who reads the reports?
# - What specific insights do funders care about?
# - Are there sensitive topics in student feedback?
#
# YOUR QUESTIONS:
# 1. 
# 2. 
# 3. 
# 4. 
# 5. 
# 6. 
# 7. 
# 8. 
# 9. 
# 10. 


---

### Reference Discovery Questions

*(Only read this after you have written your own questions above.)*

Here are the key questions and the answers you would uncover through conversation:

**Data & Format:**
- "What tool do you use for surveys?" --> Google Forms, exported as CSV
- "What fields are in the CSV?" --> `respondent_type`, `program_city`, `session_date`, `rating` (1-5), `open_text`
- "How many responses per quarter?" --> ~400-600 responses across all 15 cities
- "Are any responses in languages other than English?" --> Yes, about 15% are in Spanish

**Report & Audience:**
- "What does the quarterly report look like?" --> Executive summary, city-by-city breakdown, themes across all cities, notable quotes, recommendations
- "Who reads these reports?" --> Foundation program officers and board members
- "What do funders specifically care about?" --> Student outcomes, program reach, retention rates, qualitative impact stories

**Constraints & Sensitivity:**
- "Is there any sensitive information in the feedback?" --> Student names are pseudonymized, but feedback sometimes mentions instructors by name
- "What would success look like?" --> Cut report preparation from 3 weeks to 3 days
- "Any edge cases I should know about?" --> Some responses are very short ("good", "ok"), some are in Spanish

**Scoring yourself:**
- Did you ask about data format? (Critical)
- Did you ask about the report structure/audience? (Critical)
- Did you ask about sensitive data? (Important -- shows you think about safety)
- Did you ask about success criteria? (Important -- shows business awareness)
- Did you ask about edge cases like language or short responses? (Bonus)

---

## Phase 2: Build (20 minutes)

Now build your solution. You have the sample data below. Write your system prompt and processing code in the empty cells. Do NOT look at the reference solution until your 20 minutes are up.

In [ ]:
# Sample feedback data (use this to develop and test your prompt)

sample_feedback = [
    {
        "respondent_type": "student",
        "city": "Chicago",
        "session": "Fall 2024",
        "rating": 5,
        "text": "I learned how to make my own website! The instructor Marco was really patient when I got stuck on CSS. I wish we had more time for the final project."
    },
    {
        "respondent_type": "parent",
        "city": "Chicago",
        "session": "Fall 2024",
        "rating": 4,
        "text": "My daughter loved the program. She comes home excited to show me what she built. Only concern is the late pickup time - 6pm is hard for working parents."
    },
    {
        "respondent_type": "instructor",
        "city": "Chicago",
        "session": "Fall 2024",
        "rating": 4,
        "text": "Great cohort this session. Retention was 85%. Main challenge was varying skill levels - some kids had prior experience while others were complete beginners. Need better placement assessment."
    },
    {
        "respondent_type": "student",
        "city": "Detroit",
        "session": "Fall 2024",
        "rating": 3,
        "text": "It was ok. Sometimes the WiFi didn't work and we couldn't do anything. My favorite part was when we made games."
    },
    {
        "respondent_type": "parent",
        "city": "Detroit",
        "session": "Fall 2024",
        "rating": 5,
        "text": "Este programa es incre\u00edble. Mi hijo ahora quiere estudiar computaci\u00f3n en la universidad. Gracias por ofrecer esto gratis."
    },
    {
        "respondent_type": "student",
        "city": "Atlanta",
        "session": "Fall 2024",
        "rating": 5,
        "text": "Best after school program ever! I made an app that tracks my basketball stats. Coach Williams even uses it now."
    },
    {
        "respondent_type": "instructor",
        "city": "Atlanta",
        "session": "Fall 2024",
        "rating": 3,
        "text": "Facility issues continue - not enough laptops for all students. Had to pair students up which slowed progress. Content curriculum needs updating, some libraries we teach are deprecated."
    },
    {
        "respondent_type": "parent",
        "city": "Atlanta",
        "session": "Fall 2024",
        "rating": 4,
        "text": "Program is wonderful but communication could be better. We didn't get the schedule until the week before it started."
    }
]

print(f"Loaded {len(sample_feedback)} feedback entries")
print(f"Cities: {set(f['city'] for f in sample_feedback)}")
print(f"Respondent types: {set(f['respondent_type'] for f in sample_feedback)}")

In [ ]:
# YOUR SOLUTION: System prompt
# Write your system prompt here

my_system_prompt = """

"""


In [ ]:
# YOUR SOLUTION: Processing code
# Build your full solution here -- format the feedback, call the API, display results



In [ ]:
# YOUR SOLUTION: Additional processing (if needed)



---

### Reference Solution: Scenario 1

*(Only read this after your 20-minute build timer is up.)*

Below is a complete, runnable reference solution.

In [ ]:
# REFERENCE SOLUTION: Scenario 1 -- Education Nonprofit Feedback Analyzer
# Step 1: System prompt

feedback_system_prompt = """You are an expert program evaluator specializing in youth education nonprofits. You help organizations analyze program feedback and generate funder-ready report sections.

Your analysis must follow these rules:

PRIVACY & SENSITIVITY:
- Redact any instructor names mentioned in feedback. Replace with "[Instructor]".
- Redact any other personal names that are not pseudonyms. Replace with "[Name]".
- Do NOT include any information that could identify individual students.

LANGUAGE:
- If feedback is in Spanish or another language, translate it to English for the report.
- Note the original language in parentheses when quoting translated feedback.

OUTPUT FORMAT:
You will produce a structured report section in the following format:

## Executive Summary
A 3-4 sentence overview of key findings, average satisfaction, and top-level themes.

## Satisfaction Metrics
- Overall average rating
- Rating breakdown by respondent type (students, parents, instructors)
- Rating breakdown by city

## Key Themes
Organize findings into these categories:
1. **Curriculum & Learning** -- What are students learning? What works? What needs improvement?
2. **Logistics & Facilities** -- Scheduling, equipment, space, WiFi, transportation
3. **Instructor Quality** -- Teaching effectiveness, patience, engagement (use redacted names)
4. **Student Impact** -- Evidence of learning outcomes, enthusiasm, career aspirations
5. **Parent Satisfaction** -- Communication, scheduling concerns, perceived value

For each theme, indicate:
- Whether it is a STRENGTH or an AREA FOR IMPROVEMENT
- How many responses mention it
- A representative quote (redacted and translated if needed)

## City Highlights
For each city represented in the data, provide a 2-3 sentence summary of standout findings.

## Notable Quotes
Select 3-5 quotes that best illustrate program impact. These should be suitable for inclusion in a funder report. Redact names and translate as needed.

## Recommendations
Based on the feedback, provide 3-5 actionable recommendations prioritized by frequency and severity.

TONE:
- Professional and evidence-based, appropriate for foundation program officers and board members.
- Balanced -- highlight both strengths and areas for improvement.
- Use specific data points and quotes to support every claim.
"""

print("System prompt defined.")
print(f"Length: {len(feedback_system_prompt)} characters")

In [ ]:
# REFERENCE SOLUTION: Scenario 1
# Step 2: Format feedback data for the prompt

def format_feedback_for_prompt(feedback_list):
    """Format feedback entries into a readable string for the prompt."""
    formatted = []
    for i, entry in enumerate(feedback_list, 1):
        formatted.append(
            f"--- Feedback #{i} ---\n"
            f"Respondent Type: {entry['respondent_type']}\n"
            f"City: {entry['city']}\n"
            f"Session: {entry['session']}\n"
            f"Rating: {entry['rating']}/5\n"
            f"Response: {entry['text']}\n"
        )
    return "\n".join(formatted)

formatted_feedback = format_feedback_for_prompt(sample_feedback)
print(formatted_feedback[:500] + "\n...")

In [ ]:
# REFERENCE SOLUTION: Scenario 1
# Step 3: Generate the report

user_prompt = f"""Please analyze the following program feedback from LearnForward's Fall 2024 session and generate a quarterly report section.

There are {len(sample_feedback)} feedback responses from {len(set(f['city'] for f in sample_feedback))} cities.

FEEDBACK DATA:
{formatted_feedback}

Generate the full report section following your output format guidelines."""

print("Generating report...")
start_time = time.time()

report = run(user_prompt, system=feedback_system_prompt)

elapsed = time.time() - start_time
print(f"Report generated in {elapsed:.1f} seconds")
print(f"Report length: {len(report)} characters")
print("\n" + "="*80)
print(report)

In [ ]:
# REFERENCE SOLUTION: Scenario 1
# Step 4: Verify key behaviors

print("=" * 60)
print("VERIFICATION CHECKS")
print("=" * 60)

# Check 1: Instructor name redaction
names_to_check = ["Marco", "Williams"]
for name in names_to_check:
    if name in report:
        print(f"FAIL: '{name}' was NOT redacted from the report")
    else:
        print(f"PASS: '{name}' was properly redacted")

# Check 2: [Instructor] placeholder present
if "[Instructor]" in report:
    print("PASS: [Instructor] placeholder found in report")
else:
    print("WARN: [Instructor] placeholder not found -- check redaction approach")

# Check 3: Spanish feedback was translated
spanish_phrases = ["incre\u00edble", "computaci\u00f3n", "universidad"]
spanish_found = any(phrase in report for phrase in spanish_phrases)
if not spanish_found:
    print("PASS: Spanish text appears to be translated")
else:
    print("WARN: Some Spanish text may not be translated")

# Check 4: Key sections present
expected_sections = ["Executive Summary", "Satisfaction", "Themes", "Recommendations"]
for section in expected_sections:
    if section.lower() in report.lower():
        print(f"PASS: '{section}' section found")
    else:
        print(f"WARN: '{section}' section not found")

# Check 5: All cities mentioned
cities = ["Chicago", "Detroit", "Atlanta"]
for city in cities:
    if city in report:
        print(f"PASS: '{city}' mentioned in report")
    else:
        print(f"WARN: '{city}' not mentioned in report")

---

## Phase 3: Present (~15-20 min)

Practice presenting your solution OUT LOUD. This is as important as the code.

### Presentation Structure

**1. Recap Their Problem (1-2 min)**
> "Based on our conversation, your core challenge is manually processing hundreds of feedback responses into quarterly funder reports, which takes your team 3 weeks. Your key concerns are [X] and [Y]."

**2. Walk Through Your Solution (3-4 min)**
- Show the system prompt -- explain WHY each section exists
- Structured output matching their existing report format
- Privacy-first: automatic instructor name redaction
- Multi-language: translates Spanish feedback with attribution
- Professional tone calibrated for foundation program officers

**3. Live Demo (3-4 min)**
- Run it on sample data, point to specific sections
- Highlight: Spanish response translated, instructor names redacted with [Instructor]
- Show that output matches the funder report structure they described

**4. Address Their Specific Concerns + Risk Reduction (3-4 min)**
- **Accuracy**: "The structured format ensures consistent output. For production, we'd build an eval set from past reports."
- **Sensitive data**: "Names are automatically redacted. We can add regex as a safety net."
- **Trust**: "I'd recommend human review of quotes before including in final funder report. The AI draft saves 90% of the time, your team does the final 10%."
- **Scale**: "400-600 responses process in seconds. We could batch by city for parallel processing."

**5. Production Roadmap + Risk Reduction Summary (2-3 min)**
- **Phase 1**: Shadow mode -- generate AI report alongside manual report, compare quality
- **Phase 2**: AI generates draft, team reviews and edits (3 days instead of 3 weeks)
- **Phase 3**: Automated pipeline with QA spot-checks
- **Risk reduction**: Structured output = predictable format. Human-in-the-loop = quality assurance. Eval pipeline = catch regressions before they hit production.

**6. Next Steps**
- Simple UI (Streamlit/Gradio) for CSV upload
- Longitudinal comparison (this quarter vs. last quarter)
- Automated alerts for low-rated cities mid-session

---
---

# SCENARIO 2: Legal Tech Startup -- Contract Clause Extraction

---

## Customer Brief

> *The interviewer, playing the role of the customer, says:*

**"We're building a legal tech platform for small law firms. Our users upload contracts -- NDAs, service agreements, employment contracts -- and we want to automatically extract key clauses and flag potential issues. Right now lawyers manually review every contract which takes 30-60 minutes each. We're a Series A startup, so we need to move fast but can't afford errors that could have legal consequences. Can Claude help with this?"**

## Timer Guidance

Set your timer now:

| Phase | Duration | What to Do |
|-------|----------|------------|
| **Phase 1: Discovery** | ~15 minutes | Write discovery questions + note customer concerns |
| **Phase 2: Solo Build** | ~15-20 minutes | Extend starter code: system prompt, structured output, test on samples |
| **Phase 3: Present** | ~15-20 minutes | Walk through solution, demo live, address concerns, risk reduction |

**START YOUR TIMER NOW.**

**IMPORTANT**: Legal domain = high stakes. Listen for their concerns about false negatives and accuracy. You'll need to address these head-on in Phase 3.

---

## Phase 1: Discovery (10 minutes)

Write your discovery questions below. Think carefully about what is unique to the legal domain.

In [ ]:
# PHASE 1: DISCOVERY (10 minutes)
# Write your discovery questions here before looking at the reference questions below
#
# Think about:
# - What types of contracts? What format?
# - What specific clauses need extraction?
# - What does "flag potential issues" mean concretely?
# - What is the acceptable error rate? What kind of errors are worst?
# - How will lawyers interact with the output?
# - What happens when the AI is wrong?
#
# YOUR QUESTIONS:
# 1. 
# 2. 
# 3. 
# 4. 
# 5. 
# 6. 
# 7. 
# 8. 
# 9. 
# 10. 


---

### Reference Discovery Questions

*(Only read this after you have written your own questions above.)*

**Data & Format:**
- "What format are the contracts in when uploaded?" --> PDF and DOCX, converted to plain text on our backend
- "How long are typical contracts?" --> 5-50 pages, most are 10-20 pages
- "What types of contracts are most common?" --> NDAs (40%), service agreements (30%), employment contracts (20%), other (10%)

**Extraction Requirements:**
- "What specific clauses do you need extracted?" --> Termination, liability/indemnification, IP ownership, non-compete, payment terms, confidentiality, governing law
- "What output format works for your platform?" --> Structured JSON: clause text, clause type, risk_level (low/medium/high), and a plain-English explanation of what the clause means and any concerns
- "What makes a clause 'high risk'?" --> Unlimited liability, broad non-competes, one-sided IP assignment, auto-renewal without notice, unusual governing law

**Risk & Error Tolerance:**
- "What happens if the AI misses a clause?" --> That is the worst case. Lawyers rely on completeness. False negatives (missing a risky clause) are unacceptable.
- "Is it okay if it flags things that turn out to be fine?" --> Yes, over-flagging is much better than under-flagging. False positives are acceptable.
- "How will lawyers use this output?" --> As a first pass. The lawyer still reviews everything, but the AI highlights where to focus.

**Edge Cases:**
- "What about contracts with non-standard formatting?" --> Common. Tables, appendices, defined terms sections.
- "Do clauses ever span multiple sections?" --> Yes, especially liability. "Subject to Section 8" type references.
- "Are there defined terms that affect interpretation?" --> Yes, e.g., 'Confidential Information' might be defined very broadly in Section 1 but referenced throughout.

**Volume:**
- "How many contracts per month?" --> ~200 per firm, we have 15 firms onboarding

**Scoring yourself:**
- Did you ask about error tolerance / false positive vs. false negative tradeoff? (Critical for legal domain)
- Did you ask about specific clause types to extract? (Critical)
- Did you ask about output format for integration? (Important)
- Did you ask about edge cases like cross-references and defined terms? (Shows depth)
- Did you ask about how the lawyer uses the output (human-in-the-loop)? (Shows safety awareness)

---

## Phase 2: Build (20 minutes)

Build your solution using the sample contract below.

In [ ]:
# Sample contract text -- a realistic NDA excerpt

sample_contract = """MUTUAL NON-DISCLOSURE AGREEMENT

This Mutual Non-Disclosure Agreement ("Agreement") is entered into as of January 15, 2025 (the "Effective Date") by and between:

TechVentures Inc., a Delaware corporation with principal offices at 450 Innovation Drive, San Jose, CA 95134 ("Company A")

and

DataFlow Solutions LLC, a California limited liability company with principal offices at 789 Market Street, Suite 400, San Francisco, CA 94103 ("Company B")

(each a "Party" and collectively the "Parties")

1. DEFINITION OF CONFIDENTIAL INFORMATION

"Confidential Information" means any and all non-public, proprietary, or confidential information disclosed by either Party to the other Party, whether orally, in writing, electronically, or by inspection of tangible objects, including but not limited to: trade secrets, patents, patent applications, inventions, research and development, product plans, products, services, customers, customer lists, markets, software (including source code and object code), developments, algorithms, technology, designs, drawings, engineering, hardware configuration, marketing, finances, business plans, personnel information, and any other information designated as "confidential" or "proprietary" or that reasonably should be understood to be confidential given the nature of the information and circumstances of disclosure.

Confidential Information shall NOT include information that: (a) was publicly known at the time of disclosure; (b) becomes publicly known through no fault of the receiving Party; (c) was rightfully in the receiving Party's possession prior to disclosure; (d) is independently developed by the receiving Party without use of the Confidential Information; or (e) is rightfully obtained by the receiving Party from a third party without restriction on disclosure.

2. OBLIGATIONS OF RECEIVING PARTY

The receiving Party shall: (a) hold all Confidential Information in strict confidence; (b) not disclose Confidential Information to any third parties without the prior written consent of the disclosing Party, except to its employees, agents, and contractors who have a need to know and are bound by confidentiality obligations no less restrictive than those contained herein; (c) use the Confidential Information solely for the purpose of evaluating and pursuing a potential business relationship between the Parties (the "Purpose"); and (d) protect the Confidential Information using at least the same degree of care it uses to protect its own confidential information, but in no event less than reasonable care.

3. TERM AND TERMINATION

This Agreement shall remain in effect for a period of three (3) years from the Effective Date, unless earlier terminated by either Party upon thirty (30) days' prior written notice. The obligations of confidentiality shall survive termination of this Agreement for a period of five (5) years following the date of disclosure of the Confidential Information. Upon termination, each Party shall promptly return or destroy all Confidential Information received from the other Party, and shall certify in writing that it has done so.

4. INTELLECTUAL PROPERTY

All Confidential Information remains the exclusive property of the disclosing Party. Nothing in this Agreement grants the receiving Party any license, right, title, or interest in or to any Confidential Information, including any intellectual property rights therein. Any inventions, improvements, or derivative works created by the receiving Party based on or incorporating the disclosing Party's Confidential Information shall be the sole and exclusive property of the disclosing Party.

5. NON-SOLICITATION

During the term of this Agreement and for a period of two (2) years following its termination, neither Party shall, directly or indirectly, solicit, hire, or attempt to hire any employee, contractor, or consultant of the other Party who was involved in the exchange of Confidential Information, without the prior written consent of the other Party.

6. REMEDIES

Each Party acknowledges that any breach of this Agreement may cause irreparable harm to the other Party for which monetary damages would be an inadequate remedy. Accordingly, either Party may seek injunctive or other equitable relief in addition to any other remedies available at law or in equity, without the requirement of posting a bond or proving actual damages. The prevailing party in any action to enforce this Agreement shall be entitled to recover its reasonable attorneys' fees and costs.

7. LIMITATION OF LIABILITY

EXCEPT FOR BREACHES OF CONFIDENTIALITY OBLIGATIONS OR INTELLECTUAL PROPERTY RIGHTS, IN NO EVENT SHALL EITHER PARTY BE LIABLE TO THE OTHER FOR ANY INDIRECT, INCIDENTAL, SPECIAL, CONSEQUENTIAL, OR PUNITIVE DAMAGES ARISING OUT OF OR RELATED TO THIS AGREEMENT, REGARDLESS OF THE FORM OF ACTION OR THEORY OF LIABILITY. The total aggregate liability of either Party under this Agreement shall not exceed ONE MILLION DOLLARS ($1,000,000).

8. GOVERNING LAW AND DISPUTE RESOLUTION

This Agreement shall be governed by and construed in accordance with the laws of the State of Delaware, without regard to its conflicts of law principles. Any disputes arising out of or relating to this Agreement shall be resolved exclusively in the state or federal courts located in Wilmington, Delaware. Each Party irrevocably consents to the personal jurisdiction and venue of such courts.

9. GENERAL PROVISIONS

This Agreement constitutes the entire agreement between the Parties with respect to the subject matter hereof and supersedes all prior negotiations, representations, and agreements. This Agreement may not be amended or modified except by a written instrument signed by both Parties. The failure of either Party to enforce any provision of this Agreement shall not constitute a waiver of such provision or the right to enforce it at a later time. If any provision of this Agreement is held to be unenforceable, the remaining provisions shall remain in full force and effect.
"""

print(f"Contract length: {len(sample_contract)} characters")
print(f"Approximate pages: {len(sample_contract) / 3000:.1f}")

In [ ]:
# YOUR SOLUTION: System prompt for contract analysis

my_contract_system_prompt = """

"""


In [ ]:
# YOUR SOLUTION: Processing code



In [ ]:
# YOUR SOLUTION: Additional processing / evaluation (if needed)



---

### Reference Solution: Scenario 2

*(Only read this after your 20-minute build timer is up.)*

In [ ]:
# REFERENCE SOLUTION: Scenario 2 -- Legal Tech Contract Clause Extraction
# Step 1: System prompt

contract_system_prompt = """You are an expert legal contract analyst. You extract key clauses from contracts and assess their risk levels for review by attorneys at small law firms.

CRITICAL PRINCIPLE: You must NEVER miss a potentially risky clause. It is far better to over-flag (false positive) than to miss something (false negative). Lawyers depend on your completeness.

CLAUSES TO EXTRACT:
For each contract, identify and extract ALL of the following clause types if present:
1. Termination -- how and when the agreement can be ended
2. Liability / Indemnification -- limits on liability, indemnification obligations
3. Intellectual Property -- IP ownership, assignment, licensing
4. Non-Compete / Non-Solicitation -- restrictions on competition or hiring
5. Payment Terms -- pricing, payment schedules, penalties for late payment
6. Confidentiality -- scope of confidential information, duration of obligations
7. Governing Law -- jurisdiction, choice of law, dispute resolution

RISK ASSESSMENT:
For each extracted clause, assign a risk level:
- LOW: Standard, balanced clause with no unusual provisions
- MEDIUM: Contains provisions that warrant attorney attention but are not uncommon
- HIGH: Contains provisions that are potentially problematic and require careful review

HIGH-RISK indicators include:
- Unlimited or uncapped liability
- Broad non-compete/non-solicitation (long duration, wide scope)
- One-sided IP assignment (all derivative works belong to one party)
- Auto-renewal without adequate notice period
- Unusual or disadvantageous governing law / venue
- Confidentiality obligations that survive unreasonably long
- Waiver of jury trial or class action rights
- Broad definition of confidential information that could impede business

OUTPUT FORMAT:
Return a valid JSON object with this structure:
{
  "contract_summary": {
    "type": "type of contract",
    "parties": ["party A", "party B"],
    "effective_date": "date",
    "overall_risk": "LOW|MEDIUM|HIGH"
  },
  "clauses": [
    {
      "clause_type": "one of the 7 types above",
      "section_reference": "section number or heading",
      "extracted_text": "the relevant clause text (abbreviated if very long)",
      "risk_level": "LOW|MEDIUM|HIGH",
      "explanation": "plain-English explanation of what this clause means and any concerns",
      "flags": ["list of specific concerns if MEDIUM or HIGH risk"]
    }
  ],
  "cross_references": [
    {
      "description": "description of cross-reference between sections",
      "sections_involved": ["Section X", "Section Y"]
    }
  ],
  "missing_clauses": ["list of expected clause types NOT found in the contract"],
  "overall_notes": "brief overall assessment"
}

IMPORTANT RULES:
- Extract the ACTUAL clause text, not a paraphrase
- If a clause is very long (>500 words), extract the most critical portion and note that it continues
- Flag ANY cross-references between sections (e.g., "Subject to Section 8")
- Note defined terms and how they affect clause interpretation
- If a standard clause type is MISSING from the contract, flag it in missing_clauses
- Return ONLY valid JSON, no markdown formatting around it
"""

print("Contract analysis system prompt defined.")
print(f"Length: {len(contract_system_prompt)} characters")

In [ ]:
# REFERENCE SOLUTION: Scenario 2
# Step 2: Run extraction on the sample contract

contract_user_prompt = f"""Analyze the following contract and extract all key clauses with risk assessments.

CONTRACT TEXT:
---
{sample_contract}
---

Return the analysis as a JSON object following your output format guidelines."""

print("Analyzing contract...")
start_time = time.time()

contract_result = run(contract_user_prompt, system=contract_system_prompt)

elapsed = time.time() - start_time
print(f"Analysis completed in {elapsed:.1f} seconds")
print(f"Result length: {len(contract_result)} characters")

In [ ]:
# REFERENCE SOLUTION: Scenario 2
# Step 3: Parse and display results

try:
    analysis = json.loads(contract_result)
    print("JSON parsed successfully.\n")
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print("Raw result (first 500 chars):")
    print(contract_result[:500])
    # Try to extract JSON from markdown code blocks
    import re
    json_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', contract_result, re.DOTALL)
    if json_match:
        analysis = json.loads(json_match.group(1))
        print("\nExtracted JSON from code block successfully.")
    else:
        analysis = None
        print("\nCould not extract JSON.")

if analysis:
    # Display summary
    summary = analysis.get("contract_summary", {})
    print("=" * 60)
    print("CONTRACT SUMMARY")
    print("=" * 60)
    print(f"  Type: {summary.get('type', 'N/A')}")
    print(f"  Parties: {', '.join(summary.get('parties', []))}")
    print(f"  Effective Date: {summary.get('effective_date', 'N/A')}")
    print(f"  Overall Risk: {summary.get('overall_risk', 'N/A')}")

    # Display clauses
    print(f"\n{'=' * 60}")
    print(f"EXTRACTED CLAUSES ({len(analysis.get('clauses', []))} found)")
    print("=" * 60)
    for i, clause in enumerate(analysis.get("clauses", []), 1):
        risk_icon = {"LOW": "[LOW]", "MEDIUM": "[MED]", "HIGH": "[HIGH]"}.get(clause.get("risk_level", ""), "[???]")
        print(f"\n  {i}. {risk_icon} {clause.get('clause_type', 'Unknown')} (Section: {clause.get('section_reference', 'N/A')})")
        print(f"     Risk: {clause.get('risk_level', 'N/A')}")
        print(f"     Explanation: {clause.get('explanation', 'N/A')[:200]}")
        if clause.get("flags"):
            for flag in clause["flags"]:
                print(f"     --> FLAG: {flag}")

    # Display cross-references
    cross_refs = analysis.get("cross_references", [])
    if cross_refs:
        print(f"\n{'=' * 60}")
        print("CROSS-REFERENCES")
        print("=" * 60)
        for ref in cross_refs:
            print(f"  - {ref.get('description', 'N/A')}")
            print(f"    Sections: {', '.join(ref.get('sections_involved', []))}")

    # Display missing clauses
    missing = analysis.get("missing_clauses", [])
    if missing:
        print(f"\n{'=' * 60}")
        print("MISSING CLAUSES")
        print("=" * 60)
        for clause_type in missing:
            print(f"  - {clause_type}")

    print(f"\n{'=' * 60}")
    print("OVERALL NOTES")
    print("=" * 60)
    print(f"  {analysis.get('overall_notes', 'N/A')}")

In [ ]:
# REFERENCE SOLUTION: Scenario 2
# Step 4: Evaluate accuracy against expected extractions

expected_clauses = {
    "Confidentiality": {
        "section": "1 & 2",
        "expected_risk": "MEDIUM",
        "reason": "Very broad definition of Confidential Information -- includes 'any and all non-public' info"
    },
    "Termination": {
        "section": "3",
        "expected_risk": "MEDIUM",
        "reason": "5-year survival period for confidentiality obligations is on the longer side"
    },
    "Intellectual Property": {
        "section": "4",
        "expected_risk": "HIGH",
        "reason": "Derivative works created by receiving party become property of disclosing party -- this is one-sided and potentially very broad"
    },
    "Non-Compete / Non-Solicitation": {
        "section": "5",
        "expected_risk": "HIGH",
        "reason": "2-year non-solicitation post-termination is aggressive; covers employees, contractors, AND consultants"
    },
    "Liability / Indemnification": {
        "section": "7",
        "expected_risk": "LOW",
        "reason": "$1M cap is reasonable; standard exclusion of consequential damages with carve-out for confidentiality and IP"
    },
    "Governing Law": {
        "section": "8",
        "expected_risk": "LOW",
        "reason": "Delaware law is standard for corporate agreements"
    }
}

print("=" * 60)
print("EVALUATION: Expected vs. Extracted")
print("=" * 60)

if analysis:
    extracted_types = [c.get("clause_type", "").lower() for c in analysis.get("clauses", [])]

    for clause_type, expected in expected_clauses.items():
        # Check if this clause type was found (fuzzy match)
        found = any(clause_type.lower().split("/")[0].strip() in et or
                    clause_type.lower().split("/")[-1].strip() in et
                    for et in extracted_types)
        status = "FOUND" if found else "MISSING"
        print(f"\n  {clause_type}:")
        print(f"    Expected section: {expected['section']}")
        print(f"    Expected risk: {expected['expected_risk']}")
        print(f"    Reason: {expected['reason']}")
        print(f"    Extraction status: {status}")

    print(f"\n{'=' * 60}")
    print(f"Clauses expected: {len(expected_clauses)}")
    print(f"Clauses extracted: {len(analysis.get('clauses', []))}")
    print("Note: Extracting MORE than expected is acceptable (over-flagging is preferred).")
else:
    print("No analysis to evaluate -- JSON parsing failed.")

---

## Phase 3: Present (~15-20 min)

Practice presenting your solution OUT LOUD.

### Presentation Structure

**1. Recap Their Problem (1-2 min)**
> "Based on our conversation, you need to extract key clauses from contracts and flag risk levels. Your main concern is that false negatives -- missing a risky clause -- are unacceptable. Over-flagging is fine."

**2. Walk Through Your Solution (3-4 min)**
- Show the system prompt -- biased toward over-flagging by design (their #1 concern)
- Structured JSON output that integrates directly into their platform UI
- Cross-reference detection for clauses that interact
- Missing clause detection -- flags when expected clauses are absent

**3. Live Demo (3-4 min)**
- Walk through extracted clauses one by one
- Highlight the HIGH risk IP clause and non-solicitation clause -- explain why
- Show the output is valid, parseable JSON ready for their frontend

**4. Address Their Specific Concerns + Risk Reduction (3-4 min)**
- **False negatives**: "The prompt explicitly instructs over-flagging. We'd track recall (completeness) on a labeled test set."
- **Legal consequences**: "This is a FIRST PASS tool. The lawyer still reviews everything -- AI highlights where to focus."
- **Accuracy**: "Built-in eval compares extracted clauses against expected set. For production: 50 labeled contracts as a golden test set."
- **Scale**: "200 contracts/month per firm processes easily. Batches API cuts cost 50% for non-urgent processing."
- **Trust**: "Shadow mode first -- AI extracts, lawyers verify, we track agreement rate before going live."

**5. Production Roadmap + Risk Reduction Summary (2-3 min)**
- **Phase 1**: Shadow mode -- AI runs alongside lawyers, compare completeness
- **Phase 2**: AI extracts, lawyers focus review on flagged clauses (30 min -> 10 min per contract)
- **Phase 3**: Full pipeline with contract-type-specific prompts, feedback loop, monitoring
- **Risk reduction**: Over-flagging bias = no missed clauses. Structured output = predictable format. Confidence scoring = escalation path for ambiguous clauses.

**6. Next Steps**
- Contract comparison (redline two versions)
- Feedback loop where lawyers correct AI assessments
- Contract-type-specific prompts (NDA vs. employment vs. service agreement)

---
---

# SCENARIO 3: E-commerce Company -- Product Review Intelligence

---

## Customer Brief

> *The interviewer, playing the role of the customer, says:*

**"I'm the VP of Product at ShopSmart, a mid-size e-commerce platform. We have millions of product reviews and we're drowning in data. Our product managers spend hours reading reviews to understand what customers love and hate about products. We want to turn reviews into actionable intelligence -- what features to improve, what to highlight in marketing, and early warning signs of quality issues. Can you show us how AI could help?"**

## Timer Guidance

Set your timer now:

| Phase | Duration | What to Do |
|-------|----------|------------|
| **Phase 1: Discovery** | ~15 minutes | Write discovery questions + note customer concerns |
| **Phase 2: Solo Build** | ~15-20 minutes | Extend starter code: system prompt, structured output, test on samples |
| **Phase 3: Present** | ~15-20 minutes | Walk through solution, demo live, address concerns, risk reduction |

**START YOUR TIMER NOW.**

**IMPORTANT**: Multiple stakeholders (PM, Marketing, QA) = different outputs from the same data. Listen for what each team needs and address in Phase 3.

---

## Phase 1: Discovery (10 minutes)

Write your discovery questions below. Think about the different stakeholders and what each one needs.

In [ ]:
# PHASE 1: DISCOVERY (10 minutes)
# Write your discovery questions here before looking at the reference questions below
#
# Think about:
# - What format is the review data in?
# - Who are the different consumers of this analysis?
# - What does "actionable intelligence" mean for each team?
# - What volume are we dealing with?
# - How would this integrate into their workflow?
# - What are the tricky edge cases in reviews?
#
# YOUR QUESTIONS:
# 1. 
# 2. 
# 3. 
# 4. 
# 5. 
# 6. 
# 7. 
# 8. 
# 9. 
# 10. 


---

### Reference Discovery Questions

*(Only read this after you have written your own questions above.)*

**Data & Format:**
- "What format is the review data in?" --> JSON from our API: `product_id`, `product_name`, `category`, `rating` (1-5), `review_text`, `verified_purchase` (boolean), `date`
- "How many reviews are we talking about?" --> ~50K new reviews per week across ~10K products
- "How long are typical reviews?" --> Ranges from one word to multiple paragraphs; average is 2-3 sentences

**Stakeholders & Outputs:**
- "Who consumes this analysis?" --> Three different teams:
  - **Product Managers**: Want detailed feature-level sentiment analysis, trends over time, competitive insights
  - **Marketing**: Want positive quotes they can use in ads, feature highlights for product pages
  - **QA Team**: Want early defect alerts -- quality issues flagged before they become widespread
- "What does a PM do with this today?" --> Manually reads reviews for their product category, takes notes, shares in weekly meeting
- "What does Marketing want specifically?" --> Short, punchy quotes from verified buyers; feature callouts for product pages
- "What triggers a QA alert?" --> Multiple mentions of the same defect, safety concerns, anything that could be a recall

**Success Criteria:**
- "What would success look like?" --> Surface insights that would take a human 2 hours in under 2 minutes
- "How would you measure value?" --> Time saved + quality issues caught earlier + marketing conversion lift

**Edge Cases:**
- "What about fake or incentivized reviews?" --> Yes, we have some. Need to de-weight non-verified reviews.
- "Do reviews sometimes talk about shipping instead of the product?" --> All the time. Need to separate product feedback from logistics feedback.
- "What about sarcastic reviews?" --> Definitely. "Great product if you enjoy things that break after a week."
- "Do reviewers mention competitors?" --> Yes, comparisons like "better than Brand X" or "should have bought Brand Y instead"

**Scoring yourself:**
- Did you ask about different stakeholders and their different needs? (Critical)
- Did you ask about volume and how this needs to scale? (Important)
- Did you ask about edge cases like sarcasm, fake reviews, shipping complaints? (Shows depth)
- Did you ask about how success would be measured? (Shows business awareness)
- Did you ask about integration into their existing workflow? (Shows practical thinking)

---

## Phase 2: Build (20 minutes)

Build your solution using the sample reviews below.

In [ ]:
# Sample product reviews -- wireless headphones

sample_reviews = [
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 5,
        "review_text": "Absolutely love these headphones! The noise canceling is incredible - I use them on my daily subway commute and can't hear a thing. Battery life easily lasts my whole work week (5 days) on a single charge. Sound quality is rich and balanced, great for both music and podcasts. The only minor thing is the ear cups get a bit warm after 2+ hours.",
        "verified_purchase": True,
        "date": "2025-01-15"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 4,
        "review_text": "Great sound quality and the ANC is top notch. Compared to my old Sony WH-1000XM4s, these are just as good at noise canceling but the app is way better. Took off one star because Bluetooth connectivity drops occasionally when I walk away from my desk - seems to have shorter range than advertised.",
        "verified_purchase": True,
        "date": "2025-01-18"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 1,
        "review_text": "TERRIBLE. The left ear cup stopped working after just 3 weeks. Tried resetting, updating firmware, everything. It's clearly a hardware defect. For $299 I expected much better quality control. Now I have to deal with the return process. Save your money and get the Bose QC45 instead.",
        "verified_purchase": True,
        "date": "2025-01-20"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 2,
        "review_text": "The headphones themselves are fine I guess, but the shipping was a disaster. Package arrived 2 weeks late and the box was completely crushed. Not sure if the headphones were damaged in transit or what but there's a slight rattle in the right cup. Disappointed.",
        "verified_purchase": True,
        "date": "2025-01-22"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 5,
        "review_text": "Best headphones I've ever owned, period. The multipoint connection is a game changer - seamlessly switches between my laptop and phone. ANC is so good my coworkers have to tap me on the shoulder to get my attention lol. Comfortable enough for all-day wear at the office.",
        "verified_purchase": True,
        "date": "2025-01-25"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 3,
        "review_text": "These are fine for the price but nothing special. ANC works okay but doesn't block out my kids screaming. Sound is decent for casual listening. The touch controls on the ear cup are too sensitive and I keep accidentally pausing my music. Wish they had physical buttons instead.",
        "verified_purchase": False,
        "date": "2025-01-28"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 4,
        "review_text": "Bought these for my husband and he loves them. Great for working from home - the microphone quality on calls is surprisingly clear according to his colleagues. He did mention the headband feels a bit tight, but he has a larger head so YMMV. Overall very happy with the purchase.",
        "verified_purchase": True,
        "date": "2025-02-01"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 1,
        "review_text": "Great product if you enjoy having your left ear cup die after a month. Same issue as other reviewers. There's clearly a manufacturing defect with the left speaker. Company should issue a recall. The right side still sounds amazing which makes it even more frustrating.",
        "verified_purchase": True,
        "date": "2025-02-03"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 5,
        "review_text": "Coming from AirPods Max and I'm impressed. The ANC is comparable, sound stage is wider, and these are half the price. Fold-flat design fits perfectly in my laptop bag. The auto-pause when you take them off is a nice touch. 10/10 would recommend to anyone looking for premium wireless headphones.",
        "verified_purchase": True,
        "date": "2025-02-05"
    },
    {
        "product_id": "WH-2000X",
        "product_name": "ProSound Wireless ANC Headphones",
        "category": "Electronics > Audio > Headphones",
        "rating": 4,
        "review_text": "Solid headphones for the price point. The companion app has a really good EQ and the spatial audio feature is cool for movies. My only complaint is the carrying case feels cheap compared to what you get with Bose or Sony at this price. Also wish they came in more colors.",
        "verified_purchase": True,
        "date": "2025-02-08"
    }
]

print(f"Loaded {len(sample_reviews)} reviews")
print(f"Average rating: {sum(r['rating'] for r in sample_reviews) / len(sample_reviews):.1f}")
print(f"Verified purchases: {sum(1 for r in sample_reviews if r['verified_purchase'])}/{len(sample_reviews)}")
print(f"Date range: {min(r['date'] for r in sample_reviews)} to {max(r['date'] for r in sample_reviews)}")

In [ ]:
# YOUR SOLUTION: System prompt for review analysis

my_review_system_prompt = """

"""


In [ ]:
# YOUR SOLUTION: Processing code



In [ ]:
# YOUR SOLUTION: Additional processing / stakeholder-specific outputs (if needed)



---

### Reference Solution: Scenario 3

*(Only read this after your 20-minute build timer is up.)*

In [ ]:
# REFERENCE SOLUTION: Scenario 3 -- E-commerce Review Intelligence
# Step 1: System prompt for multi-dimensional review analysis

review_system_prompt = """You are an expert product intelligence analyst for an e-commerce platform. You analyze customer reviews to generate actionable insights for multiple stakeholder teams.

ANALYSIS FRAMEWORK:

For each batch of reviews, produce a comprehensive analysis as a JSON object with the following sections:

1. PRODUCT OVERVIEW:
   - Average rating (overall and verified-only)
   - Rating distribution (count per star level)
   - Review volume and date range

2. FEATURE-LEVEL ANALYSIS:
   Extract every product feature mentioned and for each:
   - feature_name: standardized name (e.g., "noise_canceling", "battery_life", "comfort")
   - sentiment: POSITIVE, NEGATIVE, or MIXED
   - mention_count: how many reviews mention this feature
   - summary: one-sentence summary of sentiment
   - representative_quotes: 1-2 short quotes illustrating the sentiment

3. QUALITY ALERTS (for QA team):
   Flag any potential quality or safety issues:
   - issue_description: what the problem is
   - severity: LOW / MEDIUM / HIGH / CRITICAL
   - mention_count: how many reviews report this
   - evidence: relevant review excerpts
   - recommended_action: what QA should investigate
   Severity guidelines:
   - CRITICAL: Safety issue or potential recall situation
   - HIGH: Hardware defect pattern reported by multiple reviewers
   - MEDIUM: Recurring quality complaint that affects usability
   - LOW: Minor or isolated complaints

4. MARKETING QUOTES (for Marketing team):
   Extract quotes suitable for marketing use:
   - quote: the exact text (keep it short and punchy, under 30 words)
   - source: "Verified Buyer" if verified_purchase is true, otherwise skip
   - feature_highlighted: what product feature the quote showcases
   - usability_score: 1-5 how usable this is for marketing (5 = perfect testimonial)
   Rules: ONLY include quotes from verified purchases. No negative quotes.

5. COMPETITIVE INTELLIGENCE:
   Extract any mentions of competitors:
   - competitor_name: brand/product mentioned
   - comparison_type: FAVORABLE, UNFAVORABLE, or NEUTRAL
   - context: what was being compared

6. PM SUMMARY (for Product Managers):
   - top_strengths: top 3 things customers love (with evidence)
   - top_improvements: top 3 things to improve (with evidence)
   - feature_requests: any features customers wish existed
   - trend_signals: any emerging patterns (e.g., increasing defect reports)

IMPORTANT RULES:
- Distinguish between PRODUCT feedback and LOGISTICS/SHIPPING feedback. Tag shipping complaints separately.
- Detect sarcasm and classify sentiment correctly. "Great product if you enjoy things that break" = NEGATIVE.
- Weight verified purchases more heavily than non-verified.
- Flag patterns: if 2+ reviews mention the same defect, escalate severity.
- Return ONLY valid JSON, no markdown formatting around it.
"""

print("Review analysis system prompt defined.")
print(f"Length: {len(review_system_prompt)} characters")

In [ ]:
# REFERENCE SOLUTION: Scenario 3
# Step 2: Format reviews and run analysis

def format_reviews_for_prompt(reviews):
    """Format review entries into a readable string for the prompt."""
    formatted = []
    for i, review in enumerate(reviews, 1):
        formatted.append(
            f"--- Review #{i} ---\n"
            f"Product: {review['product_name']}\n"
            f"Category: {review['category']}\n"
            f"Rating: {review['rating']}/5\n"
            f"Verified Purchase: {review['verified_purchase']}\n"
            f"Date: {review['date']}\n"
            f"Review: {review['review_text']}\n"
        )
    return "\n".join(formatted)

formatted_reviews = format_reviews_for_prompt(sample_reviews)

review_user_prompt = f"""Analyze the following {len(sample_reviews)} product reviews for the ProSound Wireless ANC Headphones and generate a comprehensive intelligence report.

REVIEW DATA:
{formatted_reviews}

Generate the full analysis as a JSON object following your output format guidelines."""

print("Analyzing reviews...")
start_time = time.time()

review_result = run(review_user_prompt, system=review_system_prompt)

elapsed = time.time() - start_time
print(f"Analysis completed in {elapsed:.1f} seconds")
print(f"Result length: {len(review_result)} characters")

In [ ]:
# REFERENCE SOLUTION: Scenario 3
# Step 3: Parse and display results by stakeholder

try:
    review_analysis = json.loads(review_result)
    print("JSON parsed successfully.\n")
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    import re
    json_match = re.search(r'```(?:json)?\s*\n(.*?)\n```', review_result, re.DOTALL)
    if json_match:
        review_analysis = json.loads(json_match.group(1))
        print("Extracted JSON from code block successfully.\n")
    else:
        review_analysis = None
        print("Could not extract JSON.")
        print("Raw result (first 1000 chars):")
        print(review_result[:1000])

if review_analysis:
    # === PRODUCT OVERVIEW ===
    overview = review_analysis.get("product_overview", {})
    print("=" * 70)
    print("PRODUCT OVERVIEW")
    print("=" * 70)
    for key, value in overview.items():
        print(f"  {key}: {value}")

    # === FEATURE ANALYSIS ===
    features = review_analysis.get("feature_level_analysis", review_analysis.get("feature_analysis", []))
    print(f"\n{'=' * 70}")
    print(f"FEATURE-LEVEL ANALYSIS ({len(features)} features detected)")
    print("=" * 70)
    for feat in features:
        sentiment_marker = {"POSITIVE": "[+]", "NEGATIVE": "[-]", "MIXED": "[~]"}.get(feat.get("sentiment", ""), "[?]")
        print(f"\n  {sentiment_marker} {feat.get('feature_name', 'N/A')} (mentioned {feat.get('mention_count', '?')}x)")
        print(f"      {feat.get('summary', 'N/A')}")

In [ ]:
# REFERENCE SOLUTION: Scenario 3
# Step 4: Display QA Alerts (stakeholder-specific view)

if review_analysis:
    alerts = review_analysis.get("quality_alerts", [])
    print("=" * 70)
    print("QA TEAM DASHBOARD: Quality Alerts")
    print("=" * 70)

    if not alerts:
        print("  No quality alerts detected.")
    else:
        # Sort by severity
        severity_order = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
        sorted_alerts = sorted(alerts, key=lambda a: severity_order.get(a.get("severity", "LOW"), 4))

        for alert in sorted_alerts:
            severity = alert.get("severity", "UNKNOWN")
            icon = {"CRITICAL": "[!!!]", "HIGH": "[!! ]", "MEDIUM": "[!  ]", "LOW": "[.  ]"}.get(severity, "[?  ]")
            print(f"\n  {icon} {severity}: {alert.get('issue_description', 'N/A')}")
            print(f"       Mentions: {alert.get('mention_count', '?')}")
            print(f"       Action: {alert.get('recommended_action', 'N/A')}")
            evidence = alert.get('evidence', [])
            if isinstance(evidence, list):
                for ev in evidence[:2]:
                    print(f"       Evidence: \"{ev[:120]}...\"" if len(str(ev)) > 120 else f"       Evidence: \"{ev}\"")
            elif isinstance(evidence, str):
                print(f"       Evidence: \"{evidence[:200]}\"")

In [ ]:
# REFERENCE SOLUTION: Scenario 3
# Step 5: Display Marketing Quotes (stakeholder-specific view)

if review_analysis:
    quotes = review_analysis.get("marketing_quotes", [])
    print("=" * 70)
    print("MARKETING TEAM: Usable Quotes")
    print("=" * 70)

    if not quotes:
        print("  No marketing-ready quotes extracted.")
    else:
        # Sort by usability score
        sorted_quotes = sorted(quotes, key=lambda q: q.get("usability_score", 0), reverse=True)
        for q in sorted_quotes:
            score = q.get('usability_score', '?')
            print(f"\n  Score {score}/5 | Feature: {q.get('feature_highlighted', 'N/A')}")
            print(f"  \"{q.get('quote', 'N/A')}\"")
            print(f"  -- {q.get('source', 'Unknown')}")

In [ ]:
# REFERENCE SOLUTION: Scenario 3
# Step 6: Display PM Summary and Competitive Intelligence

if review_analysis:
    # PM Summary
    pm = review_analysis.get("pm_summary", {})
    print("=" * 70)
    print("PRODUCT MANAGER BRIEF")
    print("=" * 70)

    print("\n  TOP STRENGTHS:")
    for s in pm.get("top_strengths", []):
        if isinstance(s, dict):
            print(f"    + {s.get('strength', s.get('feature', 'N/A'))}: {s.get('evidence', s.get('detail', 'N/A'))}")
        else:
            print(f"    + {s}")

    print("\n  TOP IMPROVEMENTS NEEDED:")
    for imp in pm.get("top_improvements", []):
        if isinstance(imp, dict):
            print(f"    - {imp.get('improvement', imp.get('feature', 'N/A'))}: {imp.get('evidence', imp.get('detail', 'N/A'))}")
        else:
            print(f"    - {imp}")

    print("\n  FEATURE REQUESTS:")
    for fr in pm.get("feature_requests", []):
        if isinstance(fr, dict):
            print(f"    * {fr.get('request', fr.get('feature', 'N/A'))}")
        else:
            print(f"    * {fr}")

    print("\n  TREND SIGNALS:")
    for ts in pm.get("trend_signals", []):
        if isinstance(ts, dict):
            print(f"    >> {ts.get('signal', ts.get('trend', 'N/A'))}")
        else:
            print(f"    >> {ts}")

    # Competitive Intelligence
    competitors = review_analysis.get("competitive_intelligence", [])
    print(f"\n{'=' * 70}")
    print("COMPETITIVE INTELLIGENCE")
    print("=" * 70)
    if not competitors:
        print("  No competitor mentions detected.")
    else:
        for comp in competitors:
            comp_type = comp.get("comparison_type", "UNKNOWN")
            icon = {"FAVORABLE": "[WIN]", "UNFAVORABLE": "[LOSE]", "NEUTRAL": "[--]"}.get(comp_type, "[??]")
            print(f"\n  {icon} vs. {comp.get('competitor_name', 'N/A')}")
            print(f"       Context: {comp.get('context', 'N/A')}")

In [ ]:
# REFERENCE SOLUTION: Scenario 3
# Step 7: Summary statistics (evaluation)

if review_analysis:
    print("=" * 70)
    print("ANALYSIS SUMMARY STATISTICS")
    print("=" * 70)

    features = review_analysis.get("feature_level_analysis", review_analysis.get("feature_analysis", []))
    alerts = review_analysis.get("quality_alerts", [])
    quotes = review_analysis.get("marketing_quotes", [])
    competitors = review_analysis.get("competitive_intelligence", [])

    print(f"\n  Reviews analyzed: {len(sample_reviews)}")
    print(f"  Features identified: {len(features)}")
    print(f"  Quality alerts raised: {len(alerts)}")
    print(f"  Marketing quotes extracted: {len(quotes)}")
    print(f"  Competitor mentions found: {len(competitors)}")

    # Feature sentiment breakdown
    pos = sum(1 for f in features if f.get('sentiment') == 'POSITIVE')
    neg = sum(1 for f in features if f.get('sentiment') == 'NEGATIVE')
    mix = sum(1 for f in features if f.get('sentiment') == 'MIXED')
    print(f"\n  Feature sentiment breakdown:")
    print(f"    Positive: {pos}")
    print(f"    Negative: {neg}")
    print(f"    Mixed: {mix}")

    # Alert severity breakdown
    for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
        count = sum(1 for a in alerts if a.get('severity') == sev)
        if count > 0:
            print(f"\n  {sev} severity alerts: {count}")

    # Key validation checks
    print(f"\n{'=' * 70}")
    print("VALIDATION CHECKS")
    print("=" * 70)

    # Check: left ear cup defect should be flagged (mentioned in 2 reviews)
    defect_flagged = any(
        "left" in str(a.get("issue_description", "")).lower() or
        "left" in str(a.get("evidence", "")).lower()
        for a in alerts
    )
    print(f"  Left ear cup defect pattern flagged: {'PASS' if defect_flagged else 'FAIL (2 reviews mention this!)'}")

    # Check: shipping complaint should be separated from product issues
    shipping_separated = any(
        "shipping" in str(f.get("feature_name", "")).lower() or
        "logistics" in str(f.get("feature_name", "")).lower() or
        "shipping" in str(a.get("issue_description", "")).lower()
        for f in features for a in alerts
    )
    # Simpler check
    shipping_in_features = any("ship" in str(f.get("feature_name", "")).lower() for f in features)
    print(f"  Shipping feedback separated from product feedback: {'PASS' if shipping_in_features else 'CHECK MANUALLY'}")

    # Check: sarcastic review correctly classified as negative
    print(f"  Sarcasm detection: Check that Review #8 ('Great product if you enjoy...') was classified as NEGATIVE")

    # Check: marketing quotes only from verified purchases
    all_verified = all(q.get("source", "") == "Verified Buyer" for q in quotes)
    print(f"  Marketing quotes all from verified buyers: {'PASS' if all_verified else 'FAIL'}")

    # Check: competitors detected
    expected_competitors = ["Sony", "Bose", "AirPods"]
    found_competitors = [c.get("competitor_name", "") for c in competitors]
    for exp in expected_competitors:
        found = any(exp.lower() in fc.lower() for fc in found_competitors)
        print(f"  Competitor '{exp}' detected: {'PASS' if found else 'MISS'}")

---

## Phase 3: Present (~15-20 min)

Practice presenting your solution OUT LOUD.

### Presentation Structure

**1. Recap Their Problem (1-2 min)**
> "Based on our conversation, your PMs spend hours reading reviews manually. You need three different outputs: feature analysis for PMs, usable quotes for Marketing, and defect alerts for QA. Your concerns are [sarcasm, fake reviews, scale]."

**2. Walk Through Your Solution (3-4 min)**
- Multi-stakeholder output from a single analysis call
- Feature-level granularity -- sentiment PER FEATURE, not just overall
- Built-in intelligence: sarcasm detection, shipping vs. product separation, competitor tracking
- Safety-first for QA: pattern detection across reviews (left ear cup defect escalated because 2+ reviews mention it)

**3. Live Demo with Stakeholder-Specific Views (3-4 min)**
- QA Dashboard: "Look, it caught the left ear cup defect pattern and flagged it as HIGH severity"
- Marketing: "These quotes are pre-filtered to verified buyers and scored for usability"
- PM Brief: "Top 3 strengths, top 3 improvements, with evidence from actual reviews"
- Competitive intelligence: "It detected mentions of Sony, Bose, and AirPods and classified the comparisons"

**4. Address Their Specific Concerns + Risk Reduction (3-4 min)**
- **Sarcasm**: "The prompt explicitly handles sarcastic reviews. We can verify with test cases."
- **Fake reviews**: "Verified purchases are weighted more heavily. Non-verified reviews are de-prioritized."
- **Scale (50K/week)**: "Batch by product, process in parallel. Batches API cuts cost 50%. Alert pipeline runs hourly for QA."
- **Trust**: "Marketing quotes require human approval before use in ads. QA alerts are recommendations, not actions."
- **Accuracy**: "Run validation checks -- did it catch the defect pattern? Did it separate shipping complaints? Did it detect all competitors?"

**5. Production Roadmap + Risk Reduction Summary (2-3 min)**
- **Phase 1**: Pilot on one product category, validate against PM's manual analysis
- **Phase 2**: Expand to all categories with per-category prompt tuning
- **Phase 3**: Real-time alerting pipeline + dashboard integration
- **Risk reduction**: Structured JSON = feeds directly into BI tools. Confidence scoring = flag uncertain analyses. Human-in-the-loop for marketing quotes. Eval pipeline tracks quality over time.

**6. Next Steps**
- Trend-over-time view (this week vs. last 4 weeks)
- Anomaly detection for sudden sentiment changes
- Category-specific prompt variants
- A/B test AI-extracted marketing quotes vs. human-selected

---
---

# Wrap-Up: Reflection and Interview Prep

---

## Reflection Questions

After completing all three scenarios, reflect on these questions:

### Patterns Across Scenarios
1. **What discovery questions worked across ALL scenarios?**
   - Data format and volume
   - Who consumes the output and what they need
   - Success criteria / definition of done
   - Edge cases and error tolerance
   - Sensitive data and safety considerations

2. **What is your go-to system prompt structure?**
   - Role assignment ("You are an expert...")
   - Task description (what to analyze, what to produce)
   - Output format specification (structured, with examples)
   - Constraints and rules (what NOT to do, edge case handling)
   - Tone and audience guidance

3. **Where did you get stuck? What took the most time?**
   - Common sticking points: output format design, edge case handling, making the system prompt specific enough
   - Time sinks: writing sample data, debugging JSON output, over-engineering the solution

### What the Interviewers Are Evaluating

| Skill | What They Look For |
|-------|--------------------|
| Discovery | Do you ask the RIGHT questions? Do you uncover hidden requirements? |
| Technical Execution | Can you build a working solution quickly? Is your prompt well-structured? |
| Communication | Can you explain your choices clearly? Do you tailor the explanation to the customer? |
| Product Thinking | Do you think about the user's workflow, not just the technology? |
| Safety Awareness | Do you consider risks, edge cases, and human-in-the-loop needs? |
| Pragmatism | Do you build something that works NOW, not a perfect system? |

## Key Takeaways for the Interview

### Phase 1 (Discovery) Makes or Breaks You
- **~15 minutes** to understand the problem. Have a mental framework:
  1. Data: What does the input look like? Can I see a sample?
  2. Output: What does the customer need to see?
  3. Users: Who will use this and how?
  4. Concerns: What are they worried about? (WRITE THESE DOWN)
  5. Success: How do we know this works?
- **Listen 70%, talk 30%.** You're a consultant, not an engineer.
- **Ask "what would make you trust this system?"** -- this is gold for Phase 3.

### Phase 2 (Build) Rewards Structure Over Cleverness
- **They give you starter code.** Read it first. Understand what's there.
- Start with the system prompt. Get it 80% right, then iterate.
- **Structured JSON output is almost always the right choice.**
- Add confidence scoring -- route low-confidence to human review.
- Test on 3+ cases. Show it works before you present.
- **Reliable classifier pattern**: system prompt with categories -> JSON output -> parse & eval.

### Phase 3 (Present) Is Where You Win
- **Recap their problem first**, not your code. "Based on our conversation..."
- **Address EVERY concern** from Phase 1 by name.
- Show the output first, then explain the code only if asked.
- **Risk reduction is the theme**: structured output, confidence scoring, eval pipeline, shadow mode, human-in-the-loop.
- End with a production roadmap: shadow mode -> partial auto -> full deploy.
- Always discuss limitations honestly -- it builds trust.

### Common Mistakes to Avoid
- Jumping to code without enough discovery
- Building a generic solution instead of one tailored to their specific concerns
- Forgetting about edge cases (language, sarcasm, sensitive data)
- Over-engineering: you have 15-20 minutes, not 20 days
- Not testing the code on sample data during build
- Presenting the solution as perfect instead of acknowledging limitations
- **Forgetting to address concerns from Phase 1 in Phase 3** -- this is the #1 miss

## Pre-Interview Checklist (2 Days Out)

### Day 1: Practice the Mechanics
- [ ] Type the Colab setup from memory in under 2 minutes (pip install, client, run helper)
- [ ] Write a system prompt with role, task, categories, JSON output format, rules in under 5 minutes
- [ ] Build the classifier pattern: system prompt -> classify() with JSON parse -> eval loop
- [ ] Run through Scenario 1 fully timed (15 + 20 + 15 = 50 min)

### Day 2: Practice the Communication
- [ ] Run through Scenario 2 or 3 fully timed
- [ ] Practice the Phase 1 discovery questions OUT LOUD (not just in your head)
- [ ] Practice the Phase 3 presentation flow OUT LOUD:
  - Recap problem + concerns
  - Walk through solution + design choices
  - Live demo
  - Address each concern + risk reduction
  - Production roadmap
- [ ] Practice saying "Based on our conversation..." to transition to presenting
- [ ] Practice saying "Here's how this reduces risk..." naturally

### Interview Day
- [ ] Colab open with API key in Secrets (key: `ANTHROPIC_API_KEY`)
- [ ] Notebook or paper ready to write down customer concerns during Discovery
- [ ] Deep breath. They want to see how you think, communicate, and build -- not perfection.

---

**You've got this. The fact that you're practicing with structured simulations puts you ahead of most candidates.**